# 01. Data Prep and Stylized Facts

This notebook builds the clean daily dataset that will be reused in the rest of the project. The economic logic is simple: before testing whether the structure of risk changed after COVID-19, we first need a clean and well-understood set of transformed series. We therefore load the 14 market variables, transform them in a way that is consistent with their financial nature, document their stylized facts, and justify the COVID break date with variance break tests on the S&P 500.

## 1. Setup

We keep the notebook light and readable by importing only the packages needed for the descriptive stage. The reusable functions are stored in `src/project2_data_utils.py` so that later notebooks can rely on exactly the same cleaned inputs and break definition.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from project2_config import (
    ANALYSIS_START,
    COVID_BREAK,
    PRICE_COLUMNS,
    YIELD_COLUMNS,
    DISPLAY_NAMES,
)
from project2_data_utils import (
    ensure_output_dirs,
    load_raw_data,
    build_aligned_returns,
    split_pre_post,
    build_stylized_facts_table,
    variance_chow_test,
    quandt_andrews_style_variance_test,
    bai_perron_style_breaks,
    plot_level_panel,
    plot_correlation_heatmaps,
    plot_acf_grid,
    plot_variance_breaks,
    save_dataframe,
    save_figure,
)

sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = "{:.4f}".format
ensure_output_dirs()

## 2. Load the workbook and inspect the raw series

We first load the Excel file exactly as provided. At this stage we do not transform anything yet: the point is to verify the raw date range, the number of columns and the amount of missing data before making any modelling choice.

In [ ]:
raw_data = load_raw_data()

print(f"Raw dataset shape: {raw_data.shape}")
print(f"Raw date range: {raw_data['date'].min().date()} to {raw_data['date'].max().date()}")
raw_data.head()

In [ ]:
missing_summary = (
    raw_data.isna()
    .mean()
    .mul(100)
    .rename("missing_pct")
    .to_frame()
    .assign(first_valid=lambda df: [raw_data[col].first_valid_index() for col in df.index])
)
missing_summary

## 3. Transform the raw series into analysis variables

The transformation follows standard financial econometrics and the course instructions:

- price and index series are converted into **daily log returns in percent**;
- **except oil**, where we use a simple percentage return because the April 2020 negative WTI settlement makes a log return undefined;
- yield series are converted into **first differences in basis points**.

This distinction matters economically. Prices scale multiplicatively, which is why log returns are natural. Yields are already rates, so taking log returns would be meaningless. We therefore use changes in basis points for the US and German 10-year yields. Oil is the only deliberate exception, because the negative WTI print in April 2020 would otherwise break the return calculation exactly in the crisis window that matters for this project.

We then apply an **aligned drop of missing values** so that all subsequent comparisons use exactly the same daily sample across assets.

In [ ]:
level_data, aligned_returns = build_aligned_returns(raw_data)
pre_covid, post_covid = split_pre_post(aligned_returns)

print(f"Aligned return sample starts on: {aligned_returns['date'].min().date()}")
print(f"Aligned return sample ends on:   {aligned_returns['date'].max().date()}")
print(f"Aligned return dataset shape:    {aligned_returns.shape}")
print(f"Pre-COVID sample shape:         {pre_covid.shape}")
print(f"Post-COVID sample shape:        {post_covid.shape}")

aligned_returns.head()

In [ ]:
transformation_table = pd.DataFrame({
    "asset": PRICE_COLUMNS + YIELD_COLUMNS,
    "display_name": [DISPLAY_NAMES[col] for col in PRICE_COLUMNS + YIELD_COLUMNS],
    "transformation": [
        "log return (%)" if col != "oil" else "simple return (%)" for col in PRICE_COLUMNS
    ] + ["first difference (bp)"] * len(YIELD_COLUMNS),
})
transformation_table

## 4. Visual inspection of the main raw market series

Before going to returns, we keep a quick look at raw levels. This is useful because structural breaks often appear first as changes in trend, volatility or co-movement in the underlying market levels. We focus on four anchor markets that will remain central later in the project: equities, oil, gold and credit.

In [ ]:
level_figure = plot_level_panel(
    level_data.loc[level_data['date'] >= pd.Timestamp(ANALYSIS_START)],
    columns=["sp500", "oil", "gold", "us_hy_bonds"],
    title="Figure 1. Main market series in levels",
)
save_figure(level_figure, "01_levels_panel.png")
plt.show()

## 5. Stylized facts before and after COVID

We now compute descriptive moments and downside risk measures separately before and after the COVID break. The goal is to see whether the distribution of daily risk changes across samples, not only whether average returns move.

The table includes:
- mean and standard deviation;
- skewness and excess kurtosis;
- Jarque-Bera and Anderson-Darling normality tests;
- 5% Value-at-Risk and Expected Shortfall;
- maximum drawdown for price-based assets.

For yield changes, maximum drawdown is left blank because drawdown is economically meaningful for cumulative wealth processes, not for daily basis-point changes.

In [ ]:
stylized_facts = build_stylized_facts_table(pre_covid, post_covid)
save_dataframe(stylized_facts, "01_stylized_facts_pre_post.csv")
stylized_facts

## 6. Correlation structure before and after COVID

The project asks whether the **structure of risk** changed, not just the volatility of each asset. The first natural check is therefore the correlation matrix. We compare the full cross-asset dependence pattern before and after the COVID break.

In [ ]:
correlation_figure = plot_correlation_heatmaps(pre_covid, post_covid)
save_figure(correlation_figure, "01_pre_post_correlation_heatmaps.png")
plt.show()

## 7. Autocorrelograms and volatility clustering

Autocorrelograms are useful here for two reasons. First, they reveal whether raw returns are close to serially uncorrelated, which is typical for liquid markets. Second, they help us see whether **volatility clustering** is present, which will motivate the GARCH models in the next notebook.

We therefore plot the ACF of raw transformed series, and then the ACF of squared series as a simple variance proxy.

In [ ]:
acf_returns_figure = plot_acf_grid(aligned_returns, "Figure 3. Autocorrelograms of transformed daily series", squared=False)
save_figure(acf_returns_figure, "01_acf_returns.png")
plt.show()

In [ ]:
acf_squared_figure = plot_acf_grid(aligned_returns, "Figure 4. Autocorrelograms of squared transformed series", squared=True)
save_figure(acf_squared_figure, "01_acf_squared_returns.png")
plt.show()

## 8. Why use 11 March 2020 as the COVID break?

The economic break date is fixed at **11 March 2020**, the day the WHO declared COVID-19 a global pandemic. Because the project should not rely on a purely narrative choice, we also check whether the S&P 500 variance proxy displays a statistically meaningful break around that date.

We use two complementary tests on **squared S&P 500 returns**, which act as a daily proxy for variance:
- a **Chow-style variance break test** imposed at 11 March 2020;
- a **Quandt-Andrews style sup-F scan** over many possible break dates;
- a **Bai-Perron style multiple-break search** implemented with `ruptures`.

In [ ]:
sp500_variance_proxy = pd.concat([pre_covid[["date", "sp500"]], post_covid[["date", "sp500"]]], ignore_index=True).set_index("date")["sp500"] ** 2

covid_break_test = variance_chow_test(sp500_variance_proxy, COVID_BREAK)
quandt_table = quandt_andrews_style_variance_test(sp500_variance_proxy)
multiple_breaks = bai_perron_style_breaks(sp500_variance_proxy, n_breaks=3, min_size=63)

covid_break_table = pd.DataFrame([covid_break_test])
quandt_top5 = quandt_table.head(5)

save_dataframe(covid_break_table, "01_chow_variance_break_at_covid.csv")
save_dataframe(quandt_top5, "01_quandt_top_breaks.csv")
save_dataframe(multiple_breaks, "01_bai_perron_style_breaks.csv")

covid_break_table

In [ ]:
print("Top 5 Quandt-Andrews style break candidates on S&P 500 variance proxy")
quandt_top5

In [ ]:
print("Bai-Perron style multiple breaks on S&P 500 variance proxy")
multiple_breaks

In [ ]:
break_figure = plot_variance_breaks(
    sp500_returns=aligned_returns.set_index('date')['sp500'],
    covid_break=COVID_BREAK,
    break_table=multiple_breaks,
)
save_figure(break_figure, "01_sp500_variance_breaks.png")
plt.show()

## 9. Save the cleaned dataset for the next notebooks

A modular project becomes much easier to maintain when every later notebook starts from the same transformed dataset. We therefore save the aligned return matrix and the two sub-samples to `outputs/project2/tables/`.

In [ ]:
save_dataframe(aligned_returns, "01_aligned_returns.csv")
save_dataframe(pre_covid, "01_pre_covid_returns.csv")
save_dataframe(post_covid, "01_post_covid_returns.csv")

print("Saved cleaned data and tables under outputs/project2/")

## 10. Main takeaways from notebook 01

At this stage, we have a clean daily panel, a disciplined pre/post split, and a first statistical justification for the COVID break date. We also know whether the post-COVID period differs in terms of tail risk, cross-asset dependence and volatility clustering. This provides the exact empirical base needed for the volatility models, DCC analysis, VAR and regime analysis in the next notebooks.